# Parte 2 · Clasificación de titulares

Esta parte toma los 384 titulares de `datos/noticias.csv` (Parte 1), los etiqueta con un LLM y entrena con esas etiquetas un Transformer ligero en español. El modelo resultante se publicará en Hugging Face Hub para que el dashboard de la Parte 3 lo cargue desde ahí.

La columna `etiqueta` del CSV de la Parte 1 dice en qué sección publicó el medio la nota, no de qué trata el titular: es una **etiqueta débil**. Por eso el entrenamiento usa las etiquetas del LLM, y la etiqueta por sección solo se usa para compararlas.

Pasos:

1. Cargar los datos.
2. Etiquetar las 384 noticias con un LLM, en lotes.
3. Comparar la etiqueta por sección con la del LLM.
4. Auditoría humana de 20 casos.
5. Split estratificado 70/15/15.
6. Baseline zero-shot sobre 40 titulares del conjunto de test.
7. Fine-tune de un Transformer ligero en español.
8. Métricas en test y comparación entre zero-shot y fine-tune.
9. Guardar el modelo y dejar listo el código para subirlo a Hugging Face Hub.

El baseline zero-shot va después del split, y no al principio, para que se evalúe sobre los mismos titulares y contra las mismas etiquetas que el fine-tune. Si se calculara antes, sobre 40 titulares cualquiera, las dos cifras no serían comparables y algunos de esos titulares podrían terminar en el entrenamiento.

## 0. Librerías y constantes

`SEMILLA` fija todo lo que depende del azar en este notebook: la muestra de la auditoría, el split y el entrenamiento. Los avisos de `transformers` se silencian para que las salidas muestren solo los resultados.

In [1]:
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from IPython.display import display
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, classification_report, cohen_kappa_score,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          PrinterCallback, Trainer, TrainingArguments, pipeline, set_seed)
from transformers.utils import logging as hf_logging

SEMILLA = 42
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
DATOS = Path("datos")

pd.set_option("display.max_colwidth", None)
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

## 1. Carga de los datos

Las 384 noticias de la Parte 1, con su etiqueta por sección.

In [2]:
noticias = pd.read_csv(DATOS / "noticias.csv", encoding="utf-8-sig")
print(f"{len(noticias)} noticias")
noticias["etiqueta"].value_counts().reindex(CATEGORIAS).rename("titulares por sección").to_frame()

384 noticias


,titulares por sección
etiqueta,
politica,92
economia,99
deportes,98
ciencia_tecnologia,95


## 2. Etiquetado con un LLM

El LLM que etiquetó es Claude (Claude Opus 5.5), en la misma sesión de trabajo en que se armó este notebook, el 18 de octubre de 2026. El notebook no puede llamar al modelo, así que el etiquetado se hizo fuera de él y el resultado quedó en `datos/etiquetas_llm.csv`. La celda de abajo lo carga y comprueba que esté completo.

Procedimiento:

- Los 384 titulares se mezclaron con la semilla 42 y se dividieron en 14 lotes: 6 de 28 y 8 de 27. Mezclarlos evita que un lote quede lleno de titulares de una misma sección.
- En cada llamada el modelo recibió un lote completo, con un identificador y el titular de cada noticia. No vio la bajada, la sección ni la URL (que lleva la sección en la ruta), para que la etiqueta dependa solo del texto que después verá el clasificador.
- Las instrucciones fueron las mismas para todos los lotes:

> Clasifica cada titular de una noticia peruana en exactamente una de estas categorías: `politica`, `economia`, `deportes`, `ciencia_tecnologia`. Usa solo el texto del titular.
>
> - `politica`: Poder Ejecutivo, Congreso, partidos, elecciones, autoridades, casos judiciales o fiscales que involucran a políticos o al Estado, y política internacional.
> - `economia`: mercados, tipo de cambio, precios, empresas, inversión, empleo, comercio, minería, energía y pesca como actividades económicas, impuestos, pensiones y consumo.
> - `deportes`: competencias, clubes, selecciones, deportistas, resultados, fichajes y transmisiones.
> - `ciencia_tecnologia`: investigación y descubrimientos, salud vista desde la ciencia, medio ambiente y clima, espacio, tecnología, internet, redes sociales, inteligencia artificial, dispositivos, videojuegos y ciberseguridad.
>
> Si un titular encaja en dos categorías, elige la del tema central: por ejemplo, una ley de pensiones va a `politica` si el titular trata del trámite o los votos, y a `economia` si trata del efecto en el bolsillo. No hay categoría "otra": si ninguna encaja del todo, elige la más cercana.

Una limitación del procedimiento: el identificador de cada titular era su posición en `noticias.csv`, y ese archivo está ordenado por sección, así que la posición se relaciona con la sección. Las etiquetas se decidieron por el texto, pero un etiquetado estrictamente ciego usaría identificadores al azar.

In [3]:
etiquetas_llm = pd.read_csv(DATOS / "etiquetas_llm.csv", encoding="utf-8-sig")
por_lote = etiquetas_llm.groupby("lote").size()

assert len(etiquetas_llm) == len(noticias) and etiquetas_llm["url"].is_unique
assert set(etiquetas_llm["url"]) == set(noticias["url"])
assert set(etiquetas_llm["etiqueta_llm"]) <= set(CATEGORIAS)
assert por_lote.between(25, 30).all()

df = (noticias.rename(columns={"etiqueta": "etiqueta_seccion"})
              .merge(etiquetas_llm[["url", "etiqueta_llm", "lote"]], on="url", validate="one_to_one"))
print(f"{len(por_lote)} lotes de entre {por_lote.min()} y {por_lote.max()} titulares")
df["etiqueta_llm"].value_counts().reindex(CATEGORIAS).rename("titulares según el LLM").to_frame()

14 lotes de entre 27 y 28 titulares


,titulares según el LLM
etiqueta_llm,
politica,97
economia,96
deportes,98
ciencia_tecnologia,93


## 3. Etiqueta por sección frente a etiqueta del LLM

La tabla cruza las dos etiquetas: las filas son la sección y las columnas, el LLM, así que la diagonal reúne las coincidencias. Además del porcentaje de acuerdo se calcula el kappa de Cohen, que descuenta el acuerdo que se daría por azar (1 es acuerdo perfecto; 0, el que daría el azar). Debajo van todos los titulares en que no coinciden.

In [4]:
coinciden = df["etiqueta_seccion"] == df["etiqueta_llm"]
print(f"Coinciden: {coinciden.sum()} de {len(df)} ({coinciden.mean():.1%})")
print(f"No coinciden: {(~coinciden).sum()}")
print(f"Kappa de Cohen: {cohen_kappa_score(df['etiqueta_seccion'], df['etiqueta_llm']):.3f}")

display(pd.crosstab(df["etiqueta_seccion"], df["etiqueta_llm"], rownames=["sección"], colnames=["LLM"],
                    margins=True, margins_name="total")
          .reindex(index=CATEGORIAS + ["total"], columns=CATEGORIAS + ["total"]))
(df.loc[~coinciden, ["titular", "seccion_original", "etiqueta_seccion", "etiqueta_llm"]]
   .sort_values(["etiqueta_seccion", "etiqueta_llm"], ignore_index=True))

Coinciden: 370 de 384 (96.4%)
No coinciden: 14
Kappa de Cohen: 0.951


LLM,politica,economia,deportes,ciencia_tecnologia,total
sección,,,,,
politica,91,1,0,0,92
economia,1,93,0,5,99
deportes,0,0,98,0,98
ciencia_tecnologia,5,2,0,88,95
total,97,96,98,93,384


,titular,seccion_original,etiqueta_seccion,etiqueta_llm
0,"Netflix despedirá a unos 1.000 empleados, el 6 % de su plantilla, según medios",tecnologia/actualidad,ciencia_tecnologia,economia
1,Menos comida y más cara: El Niño amenaza los alimentos y la pesca en Latinoamérica,tecnologia/ecologia,ciencia_tecnologia,economia
2,El Niño en Perú: no está operativo ninguno de los 18 sistemas de alertas tempranas previstos desde 2017,tecnologia/ecologia,ciencia_tecnologia,politica
3,"Demetrio Pacheco, defensor ambiental en Perú: “El Estado podría haber evitado el asesinato de mi hijo” | ENTREVISTA",tecnologia/ecologia,ciencia_tecnologia,politica
4,Trump quiere llamar “superinteligencia” a la IA y amenaza con tratar como “enemigo” a quien no lo haga,tecnologia/actualidad,ciencia_tecnologia,politica
5,Trump llama a Musk un “tesoro nacional” tras condecorarlo con galardón científico de Estados Unidos,tecnologia/actualidad,ciencia_tecnologia,politica
6,"Estados Unidos suspenderá a Microsoft y a Adobe de programa de visados, anuncia JD Vance",tecnologia/actualidad,ciencia_tecnologia,politica
7,IA y Sandbox regulatorio,economia/opinion,economia,ciencia_tecnologia
8,Indecopi se pronuncia sobre presunta filtración de datos en Interseguro,economia,economia,ciencia_tecnologia
9,Creación de agentes de IA se multiplicó por 15 en el último año: ¿cómo impacta en la productividad empresarial?,economia/dia-1,economia,ciencia_tecnologia


**Qué muestran las diferencias.** Las 14 discrepancias son casos de frontera, en que la sección y el tema del titular se separan:

- **Deportes** coincide en todos los casos.
- **Economía → ciencia y tecnología (5)**: la filtración de datos de Interseguro (tres notas que el medio publicó en economía porque la afectada es una aseguradora, y que el LLM clasifica como ciberseguridad) y dos piezas sobre inteligencia artificial.
- **Ciencia y tecnología → política (5) o economía (2)**: notas publicadas en tecnología o ecología cuyo tema central es un gobierno (dos sobre Trump, las visas de Microsoft y Adobe, los sistemas de alerta ante El Niño, el asesinato de un defensor ambiental) o un efecto económico (despidos en Netflix, el precio de los alimentos).
- **Entre política y economía (2)**: la mesa técnica entre la Unión Europea y el Minem para destrabar permisos (política → economía) y la columna «Invocación por el Perú» (economía → política), cuyo título apenas da pistas del tema.

El clasificador se entrenará con el criterio del LLM, que se fija en el tema del titular y no en la sección.

## 4. Auditoría humana (20 casos)

Para estimar qué tan confiables son las etiquetas del LLM, una persona revisa 20 casos. Se eligen 5 al azar de cada categoría asignada por el LLM (semilla 42), para que las cuatro queden representadas por igual, y se presentan mezclados.

La tabla se guarda en `datos/auditoria_humana.csv` con dos columnas para completar a mano:

- `de_acuerdo`: `si` o `no`.
- `etiqueta_correcta`: solo cuando la respuesta es `no`, la categoría que corresponde.

La celda no sobrescribe el archivo si ya existe, para no borrar respuestas ya cargadas.

**El notebook se detiene aquí hasta que la auditoría esté completa.**

In [5]:
RUTA_AUDITORIA = DATOS / "auditoria_humana.csv"

if RUTA_AUDITORIA.exists():
    auditoria = pd.read_csv(RUTA_AUDITORIA, encoding="utf-8-sig", keep_default_na=False)
    print(f"{RUTA_AUDITORIA} ya existe: se carga sin modificarlo")
else:
    auditoria = (df.groupby("etiqueta_llm").sample(n=5, random_state=SEMILLA)
                   .sample(frac=1, random_state=SEMILLA)
                   .reset_index(drop=True))
    auditoria = auditoria.assign(caso=range(1, len(auditoria) + 1), de_acuerdo="", etiqueta_correcta="")
    auditoria = auditoria[["caso", "titular", "etiqueta_llm", "de_acuerdo", "etiqueta_correcta", "url"]]
    auditoria.to_csv(RUTA_AUDITORIA, index=False, encoding="utf-8-sig")
    print(f"Guardado {RUTA_AUDITORIA}: {len(auditoria)} casos para revisar")

auditoria[["caso", "titular", "etiqueta_llm", "de_acuerdo", "etiqueta_correcta"]]

datos\auditoria_humana.csv ya existe: se carga sin modificarlo


,caso,titular,etiqueta_llm,de_acuerdo,etiqueta_correcta
0,1,Bill Gates: el verdadero peligro de la IA es “que personas malintencionadas la utilicen”,ciencia_tecnologia,si,
1,2,"Alfonso, ¿ahora qué?, una crónica de Fernando Vivas sobre López-Chau, líder de Ahora Nación",politica,si,
2,3,"Estados Unidos suspenderá a Microsoft y a Adobe de programa de visados, anuncia JD Vance",politica,si,
3,4,Videojuego peruano “Dark Rally” abre concurso para convertirte en uno de sus personajes,ciencia_tecnologia,si,
4,5,¡Por primera vez en el Clausura! Piero Cari será convocado para el duelo de Alianza Lima ante Atlético Grau,deportes,si,
5,6,‘Dibu’ Martínez enciende las alarmas: salió en camilla e inmovilizado tras un fuerte choque en la Premier League | VIDEO,deportes,si,
6,7,"El precio del petróleo de Texas baja 0,31% pendiente de las tensiones en Medio Oriente",economia,si,
7,8,“Ace Combat 8: Wings of Theve” eleva la franquicia a alturas insospechadas | Reseña,ciencia_tecnologia,si,
8,9,Francis Allison afirma que está “peleando actas” y pide al JNE y los JEE un trabajo “pulcro”,politica,si,
9,10,Flavio Figallo del Partido del Buen Gobierno: “No queremos obstruir ni evitar que se gobierne. Somos un partido de centro apoya lo que tiene que ser apoyado”,politica,si,


### Resultado de la auditoría

La celda lee las respuestas de `datos/auditoria_humana.csv` y calcula el acuerdo. Con solo 20 casos el porcentaje observado es incierto, así que se acompaña de un intervalo de confianza del 95 % (método exacto de Clopper-Pearson).

In [6]:
auditoria = pd.read_csv(RUTA_AUDITORIA, encoding="utf-8-sig", keep_default_na=False)
assert auditoria["de_acuerdo"].isin(["si", "no"]).all(), "faltan respuestas en la auditoría"

acuerdos = int((auditoria["de_acuerdo"] == "si").sum())
intervalo = binomtest(acuerdos, len(auditoria)).proportion_ci(confidence_level=0.95, method="exact")
print(f"De acuerdo con el LLM: {acuerdos} de {len(auditoria)} ({acuerdos / len(auditoria):.0%})")
print(f"Intervalo de confianza del 95%: {intervalo.low:.0%} a {intervalo.high:.0%}")
(auditoria.assign(acuerdo=auditoria["de_acuerdo"] == "si")
          .groupby("etiqueta_llm")["acuerdo"].agg(["sum", "size"])
          .rename(columns={"sum": "de acuerdo", "size": "casos"}).reindex(CATEGORIAS))

De acuerdo con el LLM: 20 de 20 (100%)
Intervalo de confianza del 95%: 83% a 100%


,de acuerdo,casos
etiqueta_llm,,
politica,5,5
economia,5,5
deportes,5,5
ciencia_tecnologia,5,5


El auditor estuvo de acuerdo con el LLM en los 20 casos. Eso no prueba que las etiquetas sean perfectas: con una muestra de 20, el intervalo es compatible con una tasa de acierto real algo más baja, como muestra la celda. Sí indica que los errores, si los hay, no son frecuentes. Las etiquetas no se modifican.

El auditor dejó dos observaciones sobre el esquema de categorías, no sobre el LLM:

- **Videojuegos (casos 4 y 8).** El auditor los ve como entretenimiento, una categoría que no existe en el esquema de cuatro. Entre las disponibles, `ciencia_tecnologia` es la más razonable. Es una limitación del esquema, no un error del LLM.
- **Inteligencia artificial y sandbox regulatorio (caso 15).** Es `ciencia_tecnologia`, pero también tiene un componente de política. Las categorías no son mutuamente excluyentes en todos los casos, y una etiqueta única obliga a elegir una.

## 5. Split estratificado 70/15/15

Los datos se dividen en entrenamiento (70 %), validación (15 %) y test (15 %), estratificando por la etiqueta del LLM para que las cuatro categorías mantengan su proporción en cada parte. Se hace en dos cortes con `train_test_split` y la semilla 42: primero se separa el test y después, del resto, la validación.

- **Entrenamiento**: ajusta los pesos del modelo.
- **Validación**: sirve para elegir la mejor época del fine-tune.
- **Test**: se usa una sola vez, al final, para las métricas.

In [7]:
resto, test = train_test_split(df, test_size=0.15, stratify=df["etiqueta_llm"], random_state=SEMILLA)
train, val = train_test_split(resto, test_size=0.15 / 0.85, stratify=resto["etiqueta_llm"], random_state=SEMILLA)

print(f"Entrenamiento: {len(train)} | Validación: {len(val)} | Test: {len(test)}")
pd.DataFrame({"entrenamiento": train["etiqueta_llm"].value_counts(),
              "validación": val["etiqueta_llm"].value_counts(),
              "test": test["etiqueta_llm"].value_counts()}).reindex(CATEGORIAS)

Entrenamiento: 268 | Validación: 58 | Test: 58


,entrenamiento,validación,test
etiqueta_llm,,,
politica,68,14,15
economia,67,15,14
deportes,68,15,15
ciencia_tecnologia,65,14,14


## 6. Baseline zero-shot (40 titulares del test)

Un modelo zero-shot clasifica sin haber visto ningún ejemplo etiquetado: recibe el titular y la lista de categorías y elige la más probable. Se usa `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, un modelo multilingüe entrenado en inferencia textual (NLI). Para cada categoría estima si el titular implica la frase «Esta noticia trata de {categoría}.» y se queda con la más probable. Las categorías se le dan en lenguaje natural («ciencia y tecnología» en vez de `ciencia_tecnologia`).

Se evalúa sobre 40 titulares del test, 10 por categoría elegidos al azar con la semilla 42, contra las etiquetas del LLM. El fine-tune se medirá después sobre estos mismos 40. La primera vez descarga unos 560 MB, y en CPU tarda varios minutos, porque cada titular pasa cuatro veces por el modelo, una por categoría.

In [8]:
NOMBRES_ZERO_SHOT = {"política": "politica", "economía": "economia",
                     "deportes": "deportes", "ciencia y tecnología": "ciencia_tecnologia"}
muestra_zs = test.groupby("etiqueta_llm").sample(n=10, random_state=SEMILLA)

zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=-1)
inicio = time.time()
resultados = zero_shot(muestra_zs["titular"].tolist(), candidate_labels=list(NOMBRES_ZERO_SHOT),
                       hypothesis_template="Esta noticia trata de {}.")
muestra_zs = muestra_zs.assign(pred_zero_shot=[NOMBRES_ZERO_SHOT[r["labels"][0]] for r in resultados])
f1_zero_shot = f1_score(muestra_zs["etiqueta_llm"], muestra_zs["pred_zero_shot"], average="macro")

print(f"{len(muestra_zs)} titulares clasificados en {time.time() - inicio:.0f} s")
print(f"F1 macro zero-shot: {f1_zero_shot:.3f}")
muestra_zs.loc[muestra_zs["pred_zero_shot"] != muestra_zs["etiqueta_llm"], ["titular", "etiqueta_llm", "pred_zero_shot"]]

C:\Users\Renat\anaconda3\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


40 titulares clasificados en 357 s
F1 macro zero-shot: 0.726


,titular,etiqueta_llm,pred_zero_shot
377,TikTok anuncia la compra directa a marcas sin salir de la ‘app’ y un chatbot con IA que recomienda productos,ciencia_tecnologia,economia
118,Indecopi se pronuncia sobre presunta filtración de datos en Interseguro,ciencia_tecnologia,economia
119,Creación de agentes de IA se multiplicó por 15 en el último año: ¿cómo impacta en la productividad empresarial?,ciencia_tecnologia,economia
218,A una etapa del objetivo: Annia Cillóniz completó el tramo entre Cusco y Puno y mantiene liderato en Caminos del Inca,deportes,politica
272,André Martínez vuelve a ganar etapa y se acerca al liderato de Eduardo Castro,deportes,politica
254,Caminos del Inca 2026: mira los tiempos acumulados hasta la etapa de Cusco en cada categoría,deportes,politica
223,El show de Raúl Orlandini entre Cusco y Puno: lidera su categoría y se acerca al podio de la general,deportes,politica
225,A falta de una etapa: Eduardo Castro mantiene liderato de Caminos del Inca seguido de cerca por André Martínez,deportes,politica
148,ProInversión planea alcanzar en el 2027 una adjudicación similar al 2024: Línea 3 y 4 del Metro de Lima y los otros proyectos en cartera,economia,politica
93,"¿Chifa, hamburguesa o chicharrón?: Ayudas estatales y competencia",economia,politica


## 7. Fine-tune de DistilBETO

**Modelo elegido: `dccuchile/distilbert-base-spanish-uncased` (DistilBETO).** Es la versión destilada de BETO, el BERT preentrenado en español por la Universidad de Chile: conserva 6 de sus 12 capas y tiene 67 millones de parámetros, frente a los 110 millones del original. Se eligió por tres razones:

- **Está preentrenado en español**, así que conoce el vocabulario de los titulares sin depender de un modelo multilingüe.
- **Entrena rápido en CPU.** En el equipo de esta tarea (Intel Core i7-1355U, sin GPU), el entrenamiento completo, con evaluación y checkpoint al final de cada época, tomó 5,7 minutos: muy por debajo del límite de 20. El tiempo de cada corrida se imprime al final de la celda.
- **Pesa poco** (unos 270 MB), lo que acelera su carga en el dashboard de la Parte 3.

Que sea *uncased* (pasa todo a minúsculas) apenas importa para clasificar por tema. El tokenizer marca como desconocidas las comillas tipográficas y algunos guiones (un 2,6 % de los tokens); como siempre los reemplaza por la misma marca, no perjudican el entrenamiento.

Configuración: titulares de hasta 64 tokens (el más largo tiene 47), 8 épocas, tasa de aprendizaje de 5e-5 con calentamiento durante el primer 10 % de los pasos, lotes de 16 y *weight decay* de 0,01. Tras cada época se evalúa en validación, y al final se conserva la época con mejor F1 macro en validación. Los checkpoints intermedios se guardan en una carpeta temporal fuera del repositorio y se borran al terminar.

In [9]:
MODELO_BASE = "dccuchile/distilbert-base-spanish-uncased"
ID_A_ETIQUETA = dict(enumerate(CATEGORIAS))
ETIQUETA_A_ID = {etiqueta: i for i, etiqueta in ID_A_ETIQUETA.items()}

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)


class Titulares(torch.utils.data.Dataset):
    def __init__(self, datos):
        self.codificados = tokenizer(datos["titular"].tolist(), truncation=True, max_length=64)
        self.etiquetas = datos["etiqueta_llm"].map(ETIQUETA_A_ID).tolist()

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, i):
        return {**{clave: valores[i] for clave, valores in self.codificados.items()}, "labels": self.etiquetas[i]}


def metricas(evaluacion):
    return {"f1_macro": f1_score(evaluacion.label_ids, evaluacion.predictions.argmax(axis=-1), average="macro")}


set_seed(SEMILLA)
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(CATEGORIAS), id2label=ID_A_ETIQUETA, label2id=ETIQUETA_A_ID)
argumentos = TrainingArguments(
    output_dir=tempfile.mkdtemp(prefix="checkpoints_"), num_train_epochs=8, learning_rate=5e-5,
    warmup_steps=0.1, weight_decay=0.01, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", seed=SEMILLA, use_cpu=True,
    report_to="none", disable_tqdm=True)
entrenador = Trainer(model=modelo, args=argumentos, train_dataset=Titulares(train), eval_dataset=Titulares(val),
                     processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                     compute_metrics=metricas)
entrenador.remove_callback(PrinterCallback)

inicio = time.time()
entrenador.train()
print(f"Entrenamiento: {(time.time() - inicio) / 60:.1f} minutos")
shutil.rmtree(argumentos.output_dir, ignore_errors=True)

registros = pd.DataFrame(entrenador.state.log_history)
historial = (registros.dropna(subset=["loss"])[["epoch", "loss"]]
             .merge(registros.dropna(subset=["eval_f1_macro"])[["epoch", "eval_loss", "eval_f1_macro"]], on="epoch")
             .rename(columns={"epoch": "época", "loss": "pérdida entrenamiento", "eval_loss": "pérdida validación",
                              "eval_f1_macro": "F1 macro validación"}))
print(f"Época elegida: {historial.loc[historial['F1 macro validación'].idxmax(), 'época']:.0f}")
historial.round(3)

Entrenamiento: 5.7 minutos
Época elegida: 4


,época,pérdida entrenamiento,pérdida validación,F1 macro validación
0,1.0,1.281,0.997,0.787
1,2.0,0.677,0.480,0.896
2,3.0,0.250,0.289,0.914
3,4.0,0.088,0.313,0.930
4,5.0,0.038,0.276,0.914
5,6.0,0.024,0.255,0.930
6,7.0,0.019,0.277,0.914
7,8.0,0.017,0.282,0.914


## 8. Métricas en test

Todo lo de esta sección se calcula sobre el conjunto de test, que el modelo no vio ni para ajustar sus pesos ni para elegir la época. La referencia son las etiquetas del LLM, así que las cifras miden cuánto se acerca cada modelo al criterio del LLM; la auditoría del paso 4 es la que respalda ese criterio.

Primero, el F1 macro y el reporte por categoría del fine-tune sobre el test completo.

In [10]:
predicciones = entrenador.predict(Titulares(test)).predictions.argmax(axis=-1)
test = test.assign(pred_fine_tune=[ID_A_ETIQUETA[i] for i in predicciones])
f1_fine_tune = f1_score(test["etiqueta_llm"], test["pred_fine_tune"], average="macro")

print(f"F1 macro del fine-tune en test ({len(test)} titulares): {f1_fine_tune:.3f}\n")
print(classification_report(test["etiqueta_llm"], test["pred_fine_tune"], labels=CATEGORIAS, digits=3, zero_division=0))

F1 macro del fine-tune en test (58 titulares): 0.929

                    precision    recall  f1-score   support

          politica      1.000     0.933     0.966        15
          economia      0.824     1.000     0.903        14
          deportes      1.000     1.000     1.000        15
ciencia_tecnologia      0.917     0.786     0.846        14

          accuracy                          0.931        58
         macro avg      0.935     0.930     0.929        58
      weighted avg      0.937     0.931     0.931        58



### Matriz de confusión

Las filas son la etiqueta del LLM y las columnas, la predicción del fine-tune: la diagonal reúne los aciertos y cada celda fuera de ella, un tipo de error. Debajo van los titulares mal clasificados.

In [11]:
matriz = pd.DataFrame(confusion_matrix(test["etiqueta_llm"], test["pred_fine_tune"], labels=CATEGORIAS),
                      index=CATEGORIAS, columns=CATEGORIAS)
aciertos = int(np.trace(matriz.to_numpy()))
n_error, real_error, pred_error = max((matriz.at[r, p], r, p) for r in CATEGORIAS for p in CATEGORIAS if r != p)
titulo = f"El fine-tune acierta {aciertos} de {len(test)} titulares de test"
if n_error:
    titulo += f"; su confusión más frecuente es {real_error} → {pred_error} ({n_error})"

fig = px.imshow(matriz, text_auto=True, title=titulo,
                color_continuous_scale=["#fcfcfb", "#86b6ef", "#2a78d6", "#104281"],
                labels={"x": "Predicción del fine-tune", "y": "Etiqueta del LLM", "color": "Titulares"})
fig.update_layout(template="simple_white", paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
                  font=dict(family="system-ui, 'Segoe UI', sans-serif", size=13, color="#0b0b0b"),
                  title=dict(font_size=17, subtitle=dict(text="Matriz de confusión en test; la diagonal son aciertos",
                                                         font=dict(size=13, color="#52514e"))),
                  margin=dict(t=110))
fig.show()

test.loc[test["pred_fine_tune"] != test["etiqueta_llm"], ["titular", "etiqueta_llm", "pred_fine_tune"]]

,titular,etiqueta_llm,pred_fine_tune
119,Creación de agentes de IA se multiplicó por 15 en el último año: ¿cómo impacta en la productividad empresarial?,ciencia_tecnologia,economia
304,"Ventas de ‘smartphones’ registran una caída del 20%, motivado por la crisis de memoria RAM",ciencia_tecnologia,economia
151,Invocación por el Perú,politica,ciencia_tecnologia
118,Indecopi se pronuncia sobre presunta filtración de datos en Interseguro,ciencia_tecnologia,economia


### Zero-shot frente a fine-tune

La comparación justa es sobre los 40 titulares en que se evaluó el zero-shot; la tabla agrega, como referencia, el fine-tune sobre el test completo. Para saber si la diferencia en esos 40 titulares es más que ruido se estima un intervalo del 95 % con bootstrap pareado: se remuestrean los 40 titulares 2.000 veces y en cada remuestreo se recalcula la diferencia de F1 entre los dos modelos.

In [12]:
muestra_zs = muestra_zs.assign(pred_fine_tune=test.loc[muestra_zs.index, "pred_fine_tune"])
f1_fine_tune_40 = f1_score(muestra_zs["etiqueta_llm"], muestra_zs["pred_fine_tune"], average="macro")

comparacion = pd.DataFrame({
    "modelo": ["zero-shot (mDeBERTa)", "fine-tune (DistilBETO)", "fine-tune (DistilBETO)"],
    "titulares evaluados": ["40 del test", "40 del test", f"test completo ({len(test)})"],
    "F1 macro": [f1_zero_shot, f1_fine_tune_40, f1_fine_tune],
    "accuracy": [accuracy_score(muestra_zs["etiqueta_llm"], muestra_zs["pred_zero_shot"]),
                 accuracy_score(muestra_zs["etiqueta_llm"], muestra_zs["pred_fine_tune"]),
                 accuracy_score(test["etiqueta_llm"], test["pred_fine_tune"])],
})

rng = np.random.default_rng(SEMILLA)
real, zs, ft = (muestra_zs[c].to_numpy() for c in ["etiqueta_llm", "pred_zero_shot", "pred_fine_tune"])
diferencias = []
for _ in range(2000):
    i = rng.integers(0, len(real), len(real))
    diferencias.append(f1_score(real[i], ft[i], labels=CATEGORIAS, average="macro", zero_division=0)
                       - f1_score(real[i], zs[i], labels=CATEGORIAS, average="macro", zero_division=0))
bajo, alto = np.percentile(diferencias, [2.5, 97.5])

print(f"Diferencia de F1 macro en los 40 (fine-tune menos zero-shot): {f1_fine_tune_40 - f1_zero_shot:+.3f}")
print(f"Intervalo del 95% por bootstrap pareado: [{bajo:+.3f}, {alto:+.3f}]")
display(comparacion.round(3))
pd.crosstab(muestra_zs["pred_zero_shot"] == muestra_zs["etiqueta_llm"],
            muestra_zs["pred_fine_tune"] == muestra_zs["etiqueta_llm"],
            rownames=["zero-shot acierta"], colnames=["fine-tune acierta"])

Diferencia de F1 macro en los 40 (fine-tune menos zero-shot): +0.199
Intervalo del 95% por bootstrap pareado: [+0.072, +0.364]


,modelo,titulares evaluados,F1 macro,accuracy
0,zero-shot (mDeBERTa),40 del test,0.726,0.725
1,fine-tune (DistilBETO),40 del test,0.925,0.925
2,fine-tune (DistilBETO),test completo (58),0.929,0.931


fine-tune acierta,False,True
zero-shot acierta,,
False,2,9
True,1,28


### Por qué el fine-tune supera al zero-shot

Sobre los mismos 40 titulares, el fine-tune obtiene un F1 macro de 0,925 y el zero-shot, de 0,726: unos 20 puntos más. El intervalo del bootstrap (de +0,07 a +0,36) no incluye el cero, así que la ventaja no es ruido de muestreo, aunque con 40 titulares su tamaño exacto es incierto. En el test completo, el fine-tune se mantiene en 0,929. En la tabla pareada, el fine-tune acierta 9 titulares que el zero-shot falla, y el zero-shot solo 1 que el fine-tune falla.

La razón principal está en qué conoce cada modelo. Las métricas miden el acuerdo con el criterio del LLM, y el fine-tune vio 268 ejemplos de ese criterio; el zero-shot solo conoce el nombre de cada categoría y tiene que adivinar dónde están sus fronteras. Sus 11 errores se concentran en tres patrones:

- **Los 5 titulares del rally Caminos del Inca fueron a `politica`.** Hablan de etapas, liderato y tramos entre ciudades sin nombrar el deporte, y el modelo no los relaciona con una competencia. El fine-tune acierta los cinco: el entrenamiento tenía 7 titulares de la misma carrera, todos de `deportes`.
- **Tres titulares de economía fueron a `politica`** cuando mencionan al Estado o a instituciones (ProInversión, las ayudas estatales, la banca).
- **Tres de ciencia y tecnología fueron a `economia`** cuando la tecnología aparece en un contexto de empresas (compras en TikTok, la filtración en Interseguro, la productividad con agentes de IA).

El fine-tune no está libre de errores. Sus 4 fallos en test son casos de frontera del mismo tipo: tres titulares de tecnología con ángulo empresarial (agentes de IA y productividad, ventas de *smartphones*, la filtración en Interseguro) que predijo como `economia`, y la columna «Invocación por el Perú», cuyo título no indica el tema.

Dos advertencias:

- **Parte de la ventaja es local.** Entrenamiento y test salen del mismo medio y de los mismos 13 días, así que el modelo vio eventos que reaparecen en el test, como el rally. Con titulares de otras semanas o de otros medios, la distancia probablemente se acorte; el zero-shot no depende de eso.
- **Costo.** El zero-shot hace cuatro pasadas por titular con un modelo de 279 millones de parámetros; el fine-tune hace una sola con uno de 67 millones. Para el dashboard, el fine-tune también es la opción más liviana.